# FitBuddy — AI Fitness Plan Generator (Hugging Face, no API key)

This notebook implements the same layered architecture as the diagram — **User → Backend → HTML Templating / Workout Logic / Admin Panel → AI Services → Plan/Nutrition Generator → Database** — but runs entirely inside Colab:

| Diagram node | This notebook |
|---|---|
| FastAPI Backend (`app/main.py`) | Plain Python functions acting as "routes" |
| HTML Templating (Jinja2 + CSS) → `index.html` / `result.html` / `all_users.html` | Jinja2 templates rendered inline with `IPython.display.HTML` |
| Gemini AI Services (API) | **Hugging Face `transformers` pipelines — open models, no API key, no billing** |
| AI Plan / Nutrition Generator (`updated_plan.py`) | `parse_plan_text()` + `generate_tip()` |
| Database Layer (SQLite + SQLAlchemy) | Same: `fitbuddy.db`, `User` and `WorkoutPlan` tables |
| Admin Panel (`all_users.html`) | `list_all_users()` + a rendered admin table |

Everything downloads a small open-source model once per session and runs locally on Colab's CPU (or GPU if you enabled one) — nothing leaves the notebook.

## 1. Install dependencies

In [ ]:
!pip install -q transformers sentencepiece accelerate sqlalchemy jinja2

## 2. AI Services — Hugging Face models (replaces the Gemini AI Services box)

Two roles, matching the diagram's split between a heavier planning model and a lighter tips model:
- **WORKOUT_MODEL** (`google/flan-t5-base`) — stands in for "Gemini 1.5 Pro – Workout Plan & Updates"
- **TIP_MODEL** (`google/flan-t5-small`) — stands in for "Gemini Flash – Nutrition/Recovery Tips"

Both are public, ungated models — no Hugging Face token needed. Swap the strings below for any other model you have access to.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Loading the model/tokenizer directly (instead of via pipeline()) avoids a
# transformers-version issue where pipeline() misresolves 'text2text-generation'
# against the hosted Inference task list and raises a KeyError.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WORKOUT_MODEL = "google/flan-t5-base"
TIP_MODEL = "google/flan-t5-small"

def load_seq2seq(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model

workout_tokenizer, workout_model = load_seq2seq(WORKOUT_MODEL)
tip_tokenizer, tip_model = load_seq2seq(TIP_MODEL)

def run_seq2seq(tokenizer, model, prompt, max_new_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Loaded:", WORKOUT_MODEL, "+", TIP_MODEL, "on", DEVICE.upper())

## 3. Database Layer — SQLite + SQLAlchemy ORM (`fitbuddy.db`)

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, ForeignKey, Text
from sqlalchemy.orm import declarative_base, relationship, sessionmaker
from datetime import datetime

Base = declarative_base()

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    name = Column(String)
    age = Column(Integer)
    weight = Column(Float)
    goal = Column(String)
    intensity = Column(String)
    created_at = Column(DateTime, default=datetime.utcnow)
    plans = relationship("WorkoutPlan", back_populates="user")

class WorkoutPlan(Base):
    __tablename__ = "plans"
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey("users.id"))
    goal = Column(String)
    intensity = Column(String)
    original_plan = Column(Text)
    updated_plan = Column(Text)
    created_at = Column(DateTime, default=datetime.utcnow)
    user = relationship("User", back_populates="plans")

engine = create_engine("sqlite:///fitbuddy.db")
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine)

print("fitbuddy.db ready with tables:", Base.metadata.tables.keys())

## 4. Workout Logic — prompt building + parsing (`routes.py` + `updated_plan.py` equivalents)

In [ ]:
import re

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def build_workout_prompt(name, age, weight, goal, intensity):
    return (
        f"Create a 7-day workout plan for {name}, age {age}, weight {weight}kg. "
        f"Goal: {goal}. Preferred intensity: {intensity}. "
        "Include exactly one line per day, formatted as 'Monday: <activity>' through "
        "'Sunday: <activity>', and make at least one day a rest day."
    )

def generate_workout_plan(name, age, weight, goal, intensity):
    prompt = build_workout_prompt(name, age, weight, goal, intensity)
    return run_seq2seq(workout_tokenizer, workout_model, prompt, max_new_tokens=400)

def parse_plan_text(text):
    """Turn free-form model output into a {day: activity} dict.
    Falls back to splitting the raw text across days if the model
    didn't follow the 'Day: activity' format closely enough."""
    parsed = {}
    for day in DAYS:
        m = re.search(rf"{day}\s*[:\-]\s*(.+)", text, re.IGNORECASE)
        if m:
            parsed[day] = m.group(1).split(".")[0].strip(" .") or "Rest / active recovery"
    if len(parsed) < 4:  # format wasn't followed well enough — fall back
        chunks = [c.strip(" .") for c in re.split(r"[.;\n]", text) if c.strip()]
        parsed = {}
        for i, day in enumerate(DAYS):
            parsed[day] = chunks[i % len(chunks)] if chunks else "Rest / active recovery"
    return parsed

def build_feedback_prompt(original_plan_text, feedback):
    return (
        f"Here is a 7-day workout plan:\n{original_plan_text}\n\n"
        f"Revise it based on this feedback: \"{feedback}\". "
        "Keep the same one-line-per-day format, 'Monday: <activity>' through 'Sunday: <activity>'."
    )

def regenerate_with_feedback(original_plan_text, feedback):
    prompt = build_feedback_prompt(original_plan_text, feedback)
    return run_seq2seq(workout_tokenizer, workout_model, prompt, max_new_tokens=400)

def build_tip_prompt(goal, tip_type):
    return f"Give one short, specific {tip_type} tip (a single sentence) for someone whose fitness goal is {goal}."

def generate_tip(goal, tip_type="nutrition"):
    prompt = build_tip_prompt(goal, tip_type)
    return run_seq2seq(tip_tokenizer, tip_model, prompt, max_new_tokens=60).strip()

## 5. "Routes" — the functions a FastAPI backend would expose

In [ ]:
def create_user_and_plan(name, age, weight, goal, intensity):
    session = SessionLocal()
    user = User(name=name, age=age, weight=weight, goal=goal, intensity=intensity)
    session.add(user)
    session.commit()
    session.refresh(user)

    raw_plan = generate_workout_plan(name, age, weight, goal, intensity)
    plan_row = WorkoutPlan(user_id=user.id, goal=goal, intensity=intensity, original_plan=raw_plan)
    session.add(plan_row)
    session.commit()
    session.refresh(plan_row)

    user_id, plan_id = user.id, plan_row.id
    session.close()
    return user_id, plan_id, raw_plan

def update_plan_with_feedback(plan_id, feedback):
    session = SessionLocal()
    plan_row = session.get(WorkoutPlan, plan_id)
    updated_text = regenerate_with_feedback(plan_row.original_plan, feedback)
    plan_row.updated_plan = updated_text
    session.commit()
    session.close()
    return updated_text

def list_all_users():
    session = SessionLocal()
    users = [(u.id, u.name, u.goal, u.intensity) for u in session.query(User).all()]
    session.close()
    return users

## 6. HTML Templating — `index.html` / `result.html` / `all_users.html` equivalents (Jinja2)

In [ ]:
from jinja2 import Template
from IPython.display import display, HTML

RESULT_TEMPLATE = Template("""
<div style='font-family:sans-serif;border:1px solid #ddd;border-radius:8px;padding:16px;max-width:480px'>
  <h3 style='margin-top:0'>{{ name }}'s Plan — {{ goal }} ({{ intensity }})</h3>
  <ul>
  {% for day, activity in plan.items() %}
    <li><b>{{ day }}:</b> {{ activity }}</li>
  {% endfor %}
  </ul>
</div>
""")

ALL_USERS_TEMPLATE = Template("""
<div style='font-family:sans-serif;max-width:480px'>
  <h3>All Users (Admin Panel)</h3>
  <table border='1' cellpadding='6' style='border-collapse:collapse;width:100%'>
    <tr><th>ID</th><th>Name</th><th>Goal</th><th>Intensity</th></tr>
    {% for u in users %}
    <tr><td>{{ u[0] }}</td><td>{{ u[1] }}</td><td>{{ u[2] }}</td><td>{{ u[3] }}</td></tr>
    {% endfor %}
  </table>
</div>
""")

def render_result_page(name, goal, intensity, plan_dict):
    display(HTML(RESULT_TEMPLATE.render(name=name, goal=goal, intensity=intensity, plan=plan_dict)))

def render_admin_page():
    display(HTML(ALL_USERS_TEMPLATE.render(users=list_all_users())))

## 7. Demo — Scenario 1: generate a plan (index.html → result.html)

In [ ]:
name, age, weight, goal, intensity = "Priya", 29, 68, "weight loss", "medium"

user_id, plan_id, raw_plan_text = create_user_and_plan(name, age, weight, goal, intensity)
plan_dict = parse_plan_text(raw_plan_text)

print("Raw model output:\n", raw_plan_text, "\n")
render_result_page(name, goal, intensity, plan_dict)

## 8. Demo — Scenario 2: feedback loop

In [ ]:
feedback_text = "more focus on cardio, and include more rest days"
updated_raw = update_plan_with_feedback(plan_id, feedback_text)
updated_dict = parse_plan_text(updated_raw)

print("Feedback:", feedback_text)
print("\nRaw updated output:\n", updated_raw, "\n")
render_result_page(name + " (updated)", goal, intensity, updated_dict)

## 9. Demo — Scenario 3: nutrition / recovery tips

In [ ]:
print("Nutrition tip:", generate_tip(goal, "nutrition"))
print("Recovery tip: ", generate_tip(goal, "recovery"))

## 10. Demo — Admin panel (all_users.html equivalent)

In [ ]:
# add one more user so the admin table has more than one row
create_user_and_plan("Alex", 34, 82, "muscle gain", "high")
render_admin_page()

## Notes

- **No API key, no internet dependency once models are downloaded** — `flan-t5-base`/`flan-t5-small` are public on the Hugging Face Hub and load without `huggingface_hub.login()`.
- **Why `AutoModelForSeq2SeqLM` instead of `pipeline()`:** on some `transformers` versions, `pipeline(\"text2text-generation\")` misresolves against the hosted Inference task list and throws a `KeyError`. Loading the tokenizer/model directly (cell 2) sidesteps that entirely and is the standard way to run flan-t5.
- **Quality vs. speed:** `flan-t5-base` is small and fast but occasionally drifts from the exact day-by-day format — `parse_plan_text()` has a fallback for that. For noticeably better instruction-following, swap `WORKOUT_MODEL` for a larger instruct model (e.g. `google/flan-t5-large`, or a 7B chat model if you have a Colab GPU), at the cost of slower generation.
- **Database persists across cells** in `fitbuddy.db` for this session; download it from the Colab file browser if you want to keep it.
- **Going back to Gemini:** the function signatures of `generate_workout_plan`, `regenerate_with_feedback`, and `generate_tip` are the seam — swap their internals for `genai.GenerativeModel(...).generate_content(...)` calls and everything downstream (parsing, DB, templates) stays the same.